In [ ]:
# === Importaciones ===
import os, json, time, zipfile, textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, f1_score,
    confusion_matrix, classification_report, balanced_accuracy_score
)
from sklearn.preprocessing import LabelEncoder


In [ ]:
# === DATOS ===
Train = pd.read_csv("T_train_final_objetivo.csv")
Test  = pd.read_csv("T_test_final_objetivo.csv")

X_train = Train.iloc[:, :-1].copy()
y_train = Train.iloc[:, -1].astype(str)
X_test  = Test.iloc[:, :-1].copy()
y_test  = Test.iloc[:, -1].astype(str)

print("Formas:", X_train.shape, X_test.shape, "| Clases (train):", sorted(pd.unique(y_train)))


In [ ]:
# === CONFIGURACIÓN ===
CONFIG = {
    "usuario_declara_desbalance": None,   # None/True/False
    "importa_distinguir_clases": True,   # True si los costes por clase importan
    "top_k": None,                        # p.ej. 3 para Top-3 accuracy; None para no usar
    "rare_threshold": 0.05,               # clases raras si < 5%

    # Grid de hiperparámetros del árbol:
    "grid": {
        "criterion": ["gini", "entropy", "log_loss"],
        "max_depth": [None, 3, 5, 7, 9, 12],
        "min_samples_leaf": [1, 3, 5, 10],
        "class_weight": [None, "balanced"]
    },
    "cv_folds": 5,
    "random_state": 0,

    # Visualización del árbol truncado (para lectura humana)
    "max_depth_visual": 3,

    # Prefijo para agregación de importancias (dummies)
    "SEP": "___",

    # Carpeta de salida
    "OUTDIR": "dt_multiclase_artifacts",

    # === Config de visualización 2D ===
    # Si dejas None, se seleccionan automáticamente las dos PCs con mayor importancia.
    "PC_X": None,
    "PC_Y": None,
    # Profundidad del árbol 2D para rectángulos (ligero = más legible)
    "viz_max_depth": 4,
    "viz_min_samples_leaf": 3,
}
OUTDIR = Path(CONFIG["OUTDIR"]); OUTDIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# === UTILIDADES ===
def diagnostico_balance_multiclase(y, rare_threshold=0.05):
    y_series = pd.Series(y)
    vc = y_series.value_counts(dropna=False).sort_index()
    n = int(vc.sum()); k = int(vc.shape[0])
    tabla = pd.DataFrame({"clase": vc.index, "n": vc.values, "pct": vc.values / n})
    n_min, n_max = tabla["n"].min(), tabla["n"].max()
    IR = (n_max / n_min) if n_min > 0 else np.inf
    if IR < 1.5:
        etiqueta = "Balance razonable (IR < 1.5)"
    elif IR < 3:
        etiqueta = "Desbalance moderado (1.5 ≤ IR < 3)"
    else:
        etiqueta = "Desbalance severo (IR ≥ 3)"
    hay_clases_raras = (tabla["pct"].min() < rare_threshold)
    recomendar_estratificar = (IR >= 1.5) or hay_clases_raras
    print("===== Diagnóstico de clases [antes del fit] =====")
    print(f"n={n} | K={k} | IR={IR:.3f} -> {etiqueta}")
    for _, row in tabla.iterrows():
        print(f"Clase {row['clase']}: n={int(row['n'])} ({row['pct']:.1%})")
    if hay_clases_raras:
        clases_raras = tabla.loc[tabla["pct"] < rare_threshold, "clase"].tolist()
        print(f"⚠︎ Clases raras (<{rare_threshold:.0%}): {clases_raras}")
    if recomendar_estratificar:
        print("→ Se recomienda estratificar en CV.")
    return {
        "tabla": tabla, "n": n, "K": k, "IR": IR,
        "etiqueta": etiqueta, "clases_raras": tabla.loc[tabla["pct"] < rare_threshold, "clase"].tolist(),
        "recomendar_estratificar": recomendar_estratificar
    }

def decidir_metricas(K, diag, config):
    if config["usuario_declara_desbalance"] is not None:
        desbalance = bool(config["usuario_declara_desbalance"])
        razon = "forzado_por_usuario"
    else:
        desbalance = (diag["IR"] >= 1.5) or (len(diag["clases_raras"]) > 0)
        razon = "diagnostico_automatico"

    print(f"\n>>> Decisión de balance: desbalance={desbalance} (razón={razon})")
    importa_costes = bool(config["importa_distinguir_clases"])
    print(f">>> Importa distinguir entre clases (costes distintos): {importa_costes}")

    if K == 2:
        scoring_cv = "f1" if (desbalance or importa_costes) else "accuracy"
        plan = {"modo": "binario", "desbalance": desbalance, "importa_costes": importa_costes}
    else:
        if importa_costes:
            scoring_cv = "f1_weighted"
            plan = {"modo": "multiclase_costes", "desbalance": desbalance}
        else:
            scoring_cv = "f1_macro" if desbalance else "accuracy"
            plan = {"modo": "multiclase_desbalance" if desbalance else "multiclase_equilibrio",
                    "desbalance": desbalance}
    print(f">>> Métrica de CV seleccionada: {scoring_cv}")
    return scoring_cv, plan

def plot_confusion(cm, clases, outpath, title="Matriz de confusión (test)"):
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm)
    ax.set_title(title)
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Real")
    ax.set_xticks(range(len(clases)))
    ax.set_yticks(range(len(clases)))
    ax.set_xticklabels(clases, rotation=45, ha="right")
    ax.set_yticklabels(clases)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center")
    fig.tight_layout()
    fig.savefig(outpath, dpi=200)
    plt.close(fig)

def plot_tree_png(modelo, feature_names, class_names, outpath, max_depth_visual=3):
    fig, ax = plt.subplots(figsize=(12, 8))
    plot_tree(
        modelo,
        feature_names=feature_names,
        class_names=class_names,
        filled=True,
        max_depth=max_depth_visual
    )
    ax.set_title("Árbol (vista truncada para legibilidad)")
    fig.tight_layout()
    fig.savefig(outpath, dpi=200)
    plt.close(fig)

def agrupar_importancias_por_prefijo(df_import, sep="___"):
    grupos = {}
    for _, row in df_import.iterrows():
        feat = str(row["feature"]); imp = float(row["importance"])
        pref = feat.split(sep)[0] if sep in feat else feat
        grupos[pref] = grupos.get(pref, 0.0) + imp
    out = pd.DataFrame({"prefijo": list(grupos.keys()), "importance_sum": list(grupos.values())})
    return out.sort_values("importance_sum", ascending=False)


In [ ]:
# === DIAGNÓSTICO + MÉTRICA ===
diag = diagnostico_balance_multiclase(y_train, rare_threshold=CONFIG["rare_threshold"])
classes = np.unique(y_train); K = len(classes)
scoring_cv, plan = decidir_metricas(K, diag, CONFIG)


In [ ]:
# === MODELO Y GRIDSEARCH ===
base = DecisionTreeClassifier(random_state=CONFIG["random_state"])
param_grid = CONFIG["grid"]

cv = StratifiedKFold(n_splits=CONFIG["cv_folds"], shuffle=True, random_state=CONFIG["random_state"])
grid = GridSearchCV(
    estimator=base,
    param_grid=param_grid,
    scoring=scoring_cv,
    cv=cv,
    n_jobs=-1,
    refit=True,
    verbose=0
)

grid.fit(X_train, y_train)

print("\n=== Mejor configuración (CV) ===")
print(grid.best_params_)
print(f"Mejor {scoring_cv}: {grid.best_score_:.4f}")

best = grid.best_estimator_
classes_ = list(best.classes_)


In [ ]:
# === PROBABILIDADES / SCORES ===
probs_train = best.predict_proba(X_train) if hasattr(best, "predict_proba") else None
probs_test  = best.predict_proba(X_test)  if hasattr(best, "predict_proba") else None

Train_out = Train.copy()
Test_out  = Test.copy()

if K == 2 and probs_train is not None:
    try:
        idx_pos = classes_.index("1")
    except ValueError:
        idx_pos = np.argmax(classes_)
    Train_out["scores"] = probs_train[:, idx_pos]
    Test_out["scores"]  = probs_test[:, idx_pos]
elif probs_train is not None:
    for i, c in enumerate(classes_):
        Train_out[f"score_{c}"] = probs_train[:, i]
        Test_out[f"score_{c}"]  = probs_test[:, i]

Train_out.to_csv(OUTDIR / "T_train_final_objetivo_scores.csv", index=False)
Test_out.to_csv(OUTDIR / "T_test_final_objetivo_scores.csv", index=False)
print("Scores guardados en carpeta de artefactos.")


In [ ]:
# === EVALUACIÓN ===
if K == 2 and probs_test is not None:
    try:
        idx_pos = classes_.index("1")
    except ValueError:
        idx_pos = np.argmax(classes_)
    p_test = probs_test[:, idx_pos]

    def evaluate_thresholds(y_true, probs, thresholds=np.arange(0.0,1.0,0.01), criterio="f1"):
        rows=[]
        y_true_bin = (pd.Series(y_true).astype(str) == classes_[idx_pos]).astype(int).to_numpy()
        for t in thresholds:
            y_pred = (probs >= t).astype(int)
            acc = accuracy_score(y_true_bin, y_pred)
            prec, rec, f1, _ = precision_recall_fscore_support(y_true_bin, y_pred, average='binary', zero_division=0)
            rows.append({"t":t,"acc":acc,"prec":prec,"rec":rec,"f1":f1})
        df = pd.DataFrame(rows)
        key = "f1" if criterio=="f1" else "acc"
        t_opt = float(df.loc[df[key].idxmax(), "t"])
        return t_opt, df

    criterio = "f1" if (plan.get("desbalance", False) or plan.get("importa_costes", False)) else "acc"
    alpha_opt, thr_df = evaluate_thresholds(y_test, p_test, criterio=criterio)
    y_pred_bin = (p_test >= alpha_opt).astype(int)
    y_true_bin = (pd.Series(y_test).astype(str) == classes_[idx_pos]).astype(int).to_numpy()

    acc = accuracy_score(y_true_bin, y_pred_bin)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true_bin, y_pred_bin, average='binary', zero_division=0)

    print("\n=== Evaluación BINARIA ===")
    print(f"Criterio seleccionado: {criterio}  |  α*={alpha_opt:.3f}")
    print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")
    print("Matriz de confusión:\n", confusion_matrix(y_true_bin, y_pred_bin, labels=[0,1]))
else:
    # Multiclase o binario sin proba: usar predict directo
    y_pred = best.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(y_test, y_pred, average="weighted", zero_division=0)
    prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(y_test, y_pred, average="macro", zero_division=0)
    bal_acc = balanced_accuracy_score(y_test, y_pred)

    print("\n=== Evaluación MULTICLASE / PRED DIRECTO ===")
    print(f"Accuracy: {acc:.4f} | F1_macro: {f1_m:.4f} | F1_weighted: {f1_w:.4f} | BalancedAccuracy: {bal_acc:.4f}")

    cm = confusion_matrix(y_test, y_pred, labels=classes_)
    print("\nMatriz de confusión (filas=verdad, cols=pred):")
    print(pd.DataFrame(cm, index=[f"true_{c}" for c in classes_], columns=[f"pred_{c}" for c in classes_]))
    from pathlib import Path
    plot_confusion(cm, classes_, Path(CONFIG["OUTDIR"]) / "matriz_confusion.png")

    report_txt = classification_report(y_test, y_pred, zero_division=0)
    with open(Path(CONFIG["OUTDIR"]) / "classification_report.txt", "w", encoding="utf-8") as f:
        f.write(report_txt)
    print("\nReporte guardado en classification_report.txt")


In [ ]:
# === IMPORTANCIAS + ÁRBOL (truncado) ===
importancias = getattr(best, "feature_importances_", None)
pc_top2 = None

if importancias is not None:
    imp = pd.DataFrame({"feature": X_train.columns, "importance": importancias}).sort_values("importance", ascending=False)
    imp.to_csv(Path(CONFIG["OUTDIR"]) / "feature_importances.csv", index=False)
    print("feature_importances.csv guardado.")

    # Agregado por prefijo (para dummies)
    try:
        agg = agrupar_importancias_por_prefijo(imp, sep=CONFIG["SEP"])
        agg.to_csv(Path(CONFIG["OUTDIR"]) / "feature_importances_por_prefijo.csv", index=False)
        print("feature_importances_por_prefijo.csv guardado.")
    except Exception as e:
        print("No se pudo agregar por prefijo:", e)

    # === Seleccionar automáticamente las dos PCs con mayor importancia ===
    pc_mask = imp["feature"].str.upper().str.startswith("PC")
    imp_pcs = imp[pc_mask]
    if imp_pcs.shape[0] >= 2:
        pc_top2 = imp_pcs.head(2)["feature"].tolist()
        print("PCs seleccionadas por importancia para la visualización 2D:", pc_top2)
    else:
        print("No hay al menos 2 PCs en las features para seleccionar por importancia.")

# Árbol (vista truncada) para lectura humana
try:
    fig_path = Path(CONFIG["OUTDIR"]) / "arbol_truncado.png"
    plot_tree_png(best, list(X_train.columns), classes_, fig_path, max_depth_visual=CONFIG["max_depth_visual"])
    print("arbol_truncado.png guardado.")
except Exception as e:
    print("No se pudo graficar el árbol:", e)


In [ ]:
# === GUARDAR MODELO, COLUMNAS ESPERADAS, RESUMEN ===
import joblib

joblib.dump(best, Path(CONFIG["OUTDIR"]) / "modelo_arbol.pkl")
print("Modelo guardado:", Path(CONFIG["OUTDIR"]) / "modelo_arbol.pkl")

with open(Path(CONFIG["OUTDIR"]) / "expected_columns.json", "w", encoding="utf-8") as f:
    json.dump({"columns": list(X_train.columns), "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")}, f, ensure_ascii=False, indent=2)

resumen = {
    "best_params": grid.best_params_,
    "cv_best_score": grid.best_score_,
    "scoring_cv": grid.scoring,
    "classes": classes_
}
with open(Path(CONFIG["OUTDIR"]) / "resumen_metricas.json", "w", encoding="utf-8") as f:
    json.dump(resumen, f, ensure_ascii=False, indent=2)

print("Artefactos clave escritos en:", Path(CONFIG["OUTDIR"]).resolve())


In [ ]:
# === RECTÁNGULOS EN PC_X vs PC_Y (auto-selección por importancia) ===
PC_X = CONFIG["PC_X"]
PC_Y = CONFIG["PC_Y"]

# Auto-selección si no se forzó en CONFIG
if (PC_X is None or PC_Y is None):
    if 'pc_top2' in globals() and pc_top2 and len(pc_top2) >= 2:
        PC_X, PC_Y = pc_top2[0], pc_top2[1]
    else:
        # Fallback: primeras dos columnas que empiecen con 'PC' o primeras dos columnas del dataset
        pc_cols = [c for c in X_train.columns if c.upper().startswith("PC")]
        if len(pc_cols) >= 2:
            PC_X, PC_Y = pc_cols[0], pc_cols[1]
        else:
            PC_X, PC_Y = X_train.columns[0], X_train.columns[1]

print(f"Usando PCs para la visualización: {PC_X} (x) vs {PC_Y} (y)")

X_all = pd.concat([X_train, X_test], axis=0, ignore_index=True)
y_all = pd.concat([y_train, y_test], axis=0, ignore_index=True)

X2 = X_all[[PC_X, PC_Y]].copy()
le = LabelEncoder()
y_enc = le.fit_transform(y_all)
classes_viz = le.classes_

clf2d = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=CONFIG["viz_max_depth"],
    min_samples_leaf=CONFIG["viz_min_samples_leaf"],
    random_state=CONFIG["random_state"]
).fit(X2, y_enc)

x_min, x_max = X2[PC_X].min() - 0.5, X2[PC_X].max() + 0.5
y_min, y_max = X2[PC_Y].min() - 0.5, X2[PC_Y].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 600), np.linspace(y_min, y_max, 600))
Z = clf2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(7,6))
# === Plot de rectángulos (todas las hojas) ===
from matplotlib.patches import Rectangle

def collect_leaf_rects(tree, bounds, node_id=0):
    """
    Devuelve una lista de tuplas: [((x0,x1),(y0,y1)), clase_idx, soporte]
    con el rectángulo de cada hoja.
    bounds = ((x_min, x_max), (y_min, y_max))
    """
    left  = tree.children_left[node_id]
    right = tree.children_right[node_id]
    feat  = tree.feature[node_id]      # 0 -> PC_X, 1 -> PC_Y
    thr   = tree.threshold[node_id]

    # ¿Es hoja?
    if left == -1 and right == -1:
        value = tree.value[node_id][0]
        cls_idx = int(np.argmax(value))
        soporte = float(value.sum())
        return [ (bounds, cls_idx, soporte) ]

    (x0, x1), (y0, y1) = bounds

    rects = []
    if feat == 0:  # corte en PC_X
        rects += collect_leaf_rects(tree, ((x0, thr), (y0, y1)), left)
        rects += collect_leaf_rects(tree, ((thr, x1), (y0, y1)), right)
    elif feat == 1:  # corte en PC_Y
        rects += collect_leaf_rects(tree, ((x0, x1), (y0, thr)), left)
        rects += collect_leaf_rects(tree, ((x0, x1), (thr, y1)), right)
    else:
        # por si acaso (árbol extraño sin split válido)
        pass
    return rects

# Límites del plano (ligeramente expandidos para ver mejor los bordes)
x_min, x_max = X2[PC_X].min() - 0.5, X2[PC_X].max() + 0.5
y_min, y_max = X2[PC_Y].min() - 0.5, X2[PC_Y].max() + 0.5

rects = collect_leaf_rects(clf2d.tree_, ((x_min, x_max), (y_min, y_max)))
print(f"Se dibujan {len(rects)} rectángulos (hojas).")

fig, ax = plt.subplots(figsize=(7, 6))

# Dibuja todos los rectángulos hoja
for (xb, yb), cls_idx, soporte in rects:
    (rx0, rx1), (ry0, ry1) = xb, yb
    w, h = (rx1 - rx0), (ry1 - ry0)
    # Colores por clase (C0, C1, C2, ...) para que coincida con el scatter
    face = f"C{cls_idx}"
    edge = "k"
    ax.add_patch(
        Rectangle((rx0, ry0), w, h, linewidth=0.7,
                  edgecolor=edge, facecolor=face, alpha=0.15)
    )

# Puntos reales encima
scatter = ax.scatter(X2[PC_X], X2[PC_Y], c=y_enc, edgecolor="k", s=24)
ax.set_xlabel(PC_X); ax.set_ylabel(PC_Y)
ax.set_title(f"Rectángulos de decisión — TODAS las hojas en {PC_X} vs {PC_Y}")

# Leyenda por clase
handles, _ = scatter.legend_elements(prop="colors")
ax.legend(handles, classes_viz, title="Clase", loc="best", frameon=True)

ax.set_xlim(x_min, x_max); ax.set_ylim(y_min, y_max)
plt.tight_layout(); plt.show()


In [ ]:
# === ZIP DE ARTEFACTOS ===
zip_path = Path(CONFIG["OUTDIR"]) / "dt_multiclase_artifacts_bundle.zip"
candidates = [
    Path(CONFIG["OUTDIR"]) / "modelo_arbol.pkl",
    Path(CONFIG["OUTDIR"]) / "expected_columns.json",
    Path(CONFIG["OUTDIR"]) / "resumen_metricas.json",
    Path(CONFIG["OUTDIR"]) / "classification_report.txt",
    Path(CONFIG["OUTDIR"]) / "feature_importances.csv",
    Path(CONFIG["OUTDIR"]) / "feature_importances_por_prefijo.csv",
    Path(CONFIG["OUTDIR"]) / "matriz_confusion.png",
    Path(CONFIG["OUTDIR"]) / "arbol_truncado.png",
    Path(CONFIG["OUTDIR"]) / "T_train_final_objetivo_scores.csv",
    Path(CONFIG["OUTDIR"]) / "T_test_final_objetivo_scores.csv",
]
present = [str(f) for f in candidates if f.exists()]
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in present:
        zf.write(f, arcname=os.path.basename(f))
print("ZIP creado en:", zip_path)
print("Incluidos:", [os.path.basename(f) for f in present])